# 03 · Preprocessing (fit-en-train)

*Data Preparation*. Tomamos el dataset construido (`churn_dataset.csv`) y lo
dejamos **listo para el modelo**: imputación, encoding y descarte de columnas.

## Metodología: ajustar SOLO en train

El split ya se definió en `02`. La regla de oro de la validación out-of-period:
**todo estadístico data-derived se aprende con el train y se aplica a train y
test por igual**. Así ninguna información del futuro (test) se filtra al
preprocessing.

Separamos las transformaciones en dos tipos:

| Tipo | Transformación | ¿Aprende del train? |
|---|---|---|
| **Estructural** (sin leakage) | drop `edad`/`provincia`, tendencias/ratios → 0 | no — son constantes/deterministas |
| **Data-derived** (fit-en-train) | mediana de `antiguedad`, vocabulario de categorías one-hot | **sí — solo del train** |

Implementamos `fit_preprocessor(train)` → aprende los parámetros, y
`apply_preprocessor(df, params)` → los aplica. El `fit` ve **solo el train del
split OOT**; el `apply` transforma todo el dataset.

> Nota: para la CV GroupKFold lo estrictamente correcto es re-fitear el
> preprocessor dentro de cada fold. Acá el único parámetro data-derived (mediana
> de `antiguedad`) afecta al 0.3% de las filas, así que el impacto es nulo y
> fiteamos una vez con el train OOT. En `05_modelling` se puede re-fitear por
> fold si se quiere rigor total.


## 0 · Setup y split

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

base = Path.cwd()
while not (base / "data" / "processed").exists() and base != base.parent:
    base = base.parent
PROC = base / "data" / "processed"
df = pd.read_csv(PROC / "churn_dataset.csv", parse_dates=["mes_obs"])
print(f"dataset crudo: {df.shape[0]:,} filas x {df.shape[1]} columnas")


def oot_split(df, v=6, test_months=4, rank_col="mes_rank"):
    """Mismo split definido en 02 (inline, auto-contenido)."""
    rank = df[rank_col]
    test_start = rank.max() - test_months + 1
    train_last = test_start - 1 - v
    return (rank <= train_last).values, (rank >= test_start).values


train_mask, test_mask = oot_split(df, v=6, test_months=4)
print(f"train (para fitear el preprocessor): {train_mask.sum():,} filas")
print(f"test (solo se transforma): {test_mask.sum():,} filas")

dataset crudo: 30,356 filas x 46 columnas
train (para fitear el preprocessor): 28,311 filas
test (solo se transforma): 803 filas


## 1 · Plan de preprocessing por columna

| Columna(s) | Acción | Motivo |
|---|---|---|
| `id_vendedor`, `mes_obs`, `mes_rank` | **conservar** (no-feature) | claves para split / GroupKFold |
| `churn` | target | — |
| `edad` | **DESCARTAR** | 58% nulos + señal ~nula + `fecha_nacimiento` sucia |
| `provincia` | **DESCARTAR** | cardinalidad 221 → sparsity / riesgo de identificar |
| `tend_*`, `monto_cv_u12`, `monto_ult_vs_media` | imputar **0** (estructural) | NaN = sin ventana previa = sin cambio |
| `antiguedad_meses` | imputar **mediana de train** | data-derived; fit-en-train |
| `sexo`, `tipo_vendedor`, `departamento` | **one-hot** con vocabulario de train | data-derived; fit-en-train |


In [2]:
ID_COLS   = ["id_vendedor", "mes_obs", "mes_rank"]
TARGET    = "churn"
DROP      = ["edad", "provincia"]
# Delta features (mt vs mt-n): NULL cuando no hay historia suficiente → 0 (estructural)
DELTA_COLS = [f"d_{m}_m{n}" for m in ("monto", "nped") for n in (1, 3, 6, 9, 12)]
FILL_ZERO = ["tend_monto_u3_vs_prev3", "tend_nped_u3_vs_prev3",
             "monto_cv_u12", "monto_ult_vs_media",
             "monto_por_prod_acum",  # SAFE_DIVIDE → NULL si n_prod_acum=0 (productos huérfanos)
             *DELTA_COLS]
CAT_OHE   = ["sexo", "tipo_vendedor", "departamento"]

## 2 · `fit_preprocessor` — aprende SOLO del train

In [3]:
def fit_preprocessor(train):
    """Aprende los parámetros data-derived usando únicamente filas de train."""
    return {
        "antiguedad_median": float(train["antiguedad_meses"].median()),
        # vocabulario de categorías visto en train (incluye 'DESCONOCIDO' si hubo NaN)
        "cat_levels": {c: sorted(train[c].fillna("DESCONOCIDO").unique().tolist())
                       for c in CAT_OHE},
    }


params = fit_preprocessor(df[train_mask])
print(f"mediana de antiguedad (train): {params['antiguedad_median']:.1f}")
for c in CAT_OHE:
    print(f"  {c}: {len(params['cat_levels'][c])} categorías aprendidas del train")

mediana de antiguedad (train): 12.0
  sexo: 3 categorías aprendidas del train
  tipo_vendedor: 3 categorías aprendidas del train
  departamento: 35 categorías aprendidas del train


## 3 · `apply_preprocessor` — transforma con los parámetros de train

Las categorías que aparezcan **solo en test** (no vistas en train) quedan como
todo-ceros en sus dummies: es el comportamiento correcto fit-en-train (el modelo
no puede usar una categoría que nunca vio entrenando).


In [4]:
def apply_preprocessor(df, params):
    d = df.drop(columns=DROP).copy()
    # estructural (sin leakage)
    d[FILL_ZERO] = d[FILL_ZERO].fillna(0)
    # data-derived (mediana de train)
    d["antiguedad_meses"] = d["antiguedad_meses"].fillna(params["antiguedad_median"])
    # one-hot con vocabulario de train (categorías fijas)
    for c in CAT_OHE:
        d[c] = pd.Categorical(d[c].fillna("DESCONOCIDO"),
                              categories=params["cat_levels"][c])
    d = pd.get_dummies(d, columns=CAT_OHE, prefix=CAT_OHE)
    dummy_cols = [col for col in d.columns if any(col.startswith(p + "_") for p in CAT_OHE)]
    d[dummy_cols] = d[dummy_cols].astype(int)
    return d


df_enc = apply_preprocessor(df, params)

# diagnóstico: filas de test con alguna categoría no vista en train
n_unseen = 0
for c in CAT_OHE:
    seen = set(params["cat_levels"][c])
    unseen = ~df.loc[test_mask, c].fillna("DESCONOCIDO").isin(seen)
    n_unseen += int(unseen.sum())
print(f"Filas de test con categoría no vista en train (quedan en ceros): {n_unseen}")
print(f"Columnas tras one-hot: {df_enc.shape[1]}")

Filas de test con categoría no vista en train (quedan en ceros): 0
Columnas tras one-hot: 82


## 4 · Validación y guardado

In [5]:
FEATURES = [c for c in df_enc.columns if c not in ID_COLS + [TARGET]]
assert df_enc[FEATURES].isna().sum().sum() == 0, "quedan nulos en features!"
assert df_enc[FEATURES].select_dtypes(include="object").shape[1] == 0, "quedan columnas object!"
print(f"Filas: {len(df_enc):,} | features: {len(FEATURES)}")
print("✓ Sin nulos, sin columnas object: listo para el modelo")

OUT = PROC / "churn_dataset_processed.csv"
df_enc.to_csv(OUT, index=False)
print(f"Guardado: {OUT}  ({df_enc.shape[0]:,} x {df_enc.shape[1]})")

Filas: 30,356 | features: 78
✓ Sin nulos, sin columnas object: listo para el modelo


Guardado: /home/carlos/projects/glamour/customer-churn-prediction/clean/data/processed/churn_dataset_processed.csv  (30,356 x 82)


## 5 · Resumen

- **Fit-en-train:** la mediana de `antiguedad` y el vocabulario de categorías se
  aprenden **solo con el train del split OOT** y se aplican a todo el dataset.
- **Estructural (sin leakage):** drops + tendencias/ratios → 0.
- **Conservadas no-feature:** `id_vendedor`, `mes_obs`, `mes_rank`, `churn`.
- Categorías no vistas en train → dummies en ceros (correcto).

**Salida:** `clean/data/processed/churn_dataset_processed.csv`.

**Próximo paso:** `04_feature_engineering&selection` o `05_modelling`.
